## Great, now that we discussed a little let's continue

## Task 1
Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


Reproducibility is achieved by everyone using the same pipeline for the task at hand i.e. **nf-core/rnaseq** and passing a samplesheet as well as the FASTQ files to the pipeline. This way all differences between runs are explicitly logged in the samplesheet.

## Task 2
You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

The preferred pipeline for this task is **nf-core/rnaseq**. In its last step, quality control and reporting is done.

The tools used are: RSeqQC (TODO), Qualimap (TODO), dupRadar (TODO), Preseq (TODO), DESeq2 (TODO), Kraken2/Sylph (TODO)

## Task 3
Analyze the 2 files using an nf-core pipeline.

What does this pipeline do?

Which are the main tools that will be used in the pipeline?

**nf-core/rnaseq**:
1. Preprocessing
2. Genome alignment and quantification
3. Post-processing 
4. Quality control and reporting

## Task 4
As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

In [18]:
import pandas as pd

# add columns strandedness and seq_platform to the samplesheet provided in the nf-core/fetchngs pipeline
samplesheet = pd.read_csv("SRA_runs/samplesheet/samplesheet.csv")

samplesheet_edited = pd.DataFrame()

# columns to keep from the samplesheet
sample = samplesheet["sample"]
fastq_1 = samplesheet["fastq_1"]
fastq_2 = samplesheet["fastq_2"]

# strandedness column
strandedness = ["auto"] * len(samplesheet)

# seq_platform column
seq_platform = ["ILLUMINA"] * len(samplesheet)

samplesheet_edited["sample"] = sample
samplesheet_edited["fastq_1"] = fastq_1
samplesheet_edited["fastq_2"] = fastq_2
samplesheet_edited["strandedness"] = strandedness
samplesheet_edited["seq_platform"] = seq_platform

print(samplesheet_edited.columns.tolist())
print(samplesheet_edited)

samplesheet_edited.to_csv("samplesheet_edited.csv", index=False)

['sample', 'fastq_1', 'fastq_2', 'strandedness', 'seq_platform']
        sample                                            fastq_1  \
0  SRX19144486  ./SRA_runs/fastq/SRX19144486_SRR23195516_1.fas...   
1  SRX19144488  ./SRA_runs/fastq/SRX19144488_SRR23195511_1.fas...   

                                             fastq_2 strandedness seq_platform  
0  ./SRA_runs/fastq/SRX19144486_SRR23195516_2.fas...         auto     ILLUMINA  
1  ./SRA_runs/fastq/SRX19144488_SRR23195511_2.fas...         auto     ILLUMINA  


In [ ]:
# post here the command you used to run nf-core/rnaseq

# TODO run on just the 19th chromosome?
!nextflow run nf-core/rnaseq -r 3.27.0 -profile docker --outdir rnaseq_out/ --input samplesheet_edited.csv --fasta GCF_000001635.27_GRCm39_genomic.fna --gff GCF_000001635.27_GRCm39_genomic.gff --aligner hisat2 --skip_alignment true --skip_pseudo_alignment true

Explain all the parameters you set and why you set them in this way.



**Parameters:**
- -r: version
- --input: samplesheet with added columns for strandedness (auto) and seq_platform (illumina)
- --fasta: reference genome from the NCBI
- --gtf: file containing genome annotations 
- --outdir: directory for the output
- -profile docker: run pipeline in a container
- --aligner: specifies the alignment algorithm
- --skip_alignment: skips all of the alignment-based processes within the pipeline
- --skip_pseudo_alignment: skip all of the pseudoalignment-based processes within the pipeline

## Browsing the results

## Task 5
How did the pipeline perform?

**Side note:** The following description of performance etc. refers to the report provided on the start of day 3 instead of my own report since my run eventually failed at home hence it includes all samples instead of just the two.  


**Performance:**
- the run started at 29-Sep-2026 16:10:25 was successful
- CPU-hours: 103.8
- in real time: approx. 1 hour and 45 mins
- according to the CPU usage plot in the pipeline report HTML (see below), the STAR alignment was the most CPU intensive step, followed by the samtools-sort step and the samtools-sort-qualimap
- in terms of execution time the job duration plot in the pipeline report HTML (see below), reports an average execution time of 13.5 min for the STAR alignment and approx. 16 min for the step SALMON_INDEX

![cpu](performance_rnaseq_cpu.png)

![time](rnaseq_job_time.png)


## Task 6
Explain the quality control steps. Are you happy with the quality and why. If not, why not.
Please give additional information on : 
- ribosomal rRNA
- Duplication
- GC content

What are the possible steps that could lead to poorer results?

**Quality control steps of RNAseq:**

1. Tool: RSeQC
    - Assessment of sequencing quality, mapping quality, library complexity, nucleotide composition bias, GC bias, read distribution, transcript coverage, strand specificity, RNA integrity, sequencing saturation, splice junction annotation, etc.
2. Tool: Preseq (disabled by default; enable with --skip_preseq false)
    - for estimating the complexity of a genomic sequencing library 
    - estimates the number of redundant reads from a given sequencing depth and how many will be expected from additional sequencing 
3. Tool: Qualimap
    - provides GUI to facilitate the quality control of alignment sequencing data
    - *TODO* did we use this?
4. Tool: dupRadar
    - assesses the duplication rates in RNAseq datasets
5. Tool: featureCounts
    - *TODO*
6. Tool: DESeq2
    - differential gene expression analysis
    - *TODO* did we use this?
7. Tools: Kraken2 with Bracken or Sylph (unaligned by default); optional
    - Contamination detection on selected screening reads 
8. Tool: MultiGC
    - *TODO*



**Quality:**

The two following plots document the general quality of the RNAseq pipeline's output. First, the per sequence quality scores after trimming and second, the sequence counts after trimming. In the first plot, we can see *TODO*

![fastqc_seq_qual](rnaseq_out_allsamples/multiqc/star_salmon/multiqc_report_plots/png/fastqc_trimmed_per_sequence_quality_scores_plot.png)

![fastqc_seq_counts](rnaseq_out_allsamples/multiqc/star_salmon/multiqc_report_plots/png/fastqc_trimmed_sequence_counts_plot-pct.png)


**ribosomal rRNA:**

Makes it harder to observe *TODO*
The more the ribosomal rRNA there is, the worse is the quality of the result.

*TODO*

![featureCounts_rRNA](rnaseq_out_allsamples/multiqc/star_salmon/multiqc_report_plots/png/featurecounts_biotype_plot-pct.png)

**Duplication:**

The more duplication there is e.g. from PCR, the less genes are actually covered with the same amount of sequences.

*TODO*

![fastqc_dup](rnaseq_out_allsamples/multiqc/star_salmon/multiqc_report_plots/png/fastqc_trimmed_sequence_duplication_levels_plot.png)


**GC content:**


Per sequence GC content refers to *TODO*

Contamination that can lead to biases. Resolved through GC normalization.


The following plot from the MultiQC report shows the per sequence GC content after trimming. There we can see that as expected the GC content follows a Gaussian curve however there are two notable peaks, one around 50% and one around 85%. This could mean *TODO*.

![fastqc_gc_content](rnaseq_out_allsamples/multiqc/star_salmon/multiqc_report_plots/png/fastqc_trimmed_per_sequence_gc_content_plot_Counts.png)


**Steps for poorer results:**

*TODO*

Would you exclude any samples? If yes, which and why?

Samples with reads that have very high GC content or very high ribosomal rRNA need to be excluded.

Samples that need to be excluded:
- *TODO*

## Task 7
What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?

The 